<a href="https://colab.research.google.com/github/mubashir-dev751/starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mubashir-dev751/starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### 1. Rule in Plain Words
A piece of content is prioritized for a refresh if it has proven traffic volume (high impressions), is stale (older than 180 days since last update), and has an average search position that has slipped past page one (rank between 11 and 30).

### 2. Reason Codes and Actions
- **Primary Action**: `REFRESH_CONTENT`
- **Reason Codes**:
  - `STALE_HIGH_IMPRESSIONS`: High impressions but hasn't been updated in over 180 days.
  - `STRIKING_DISTANCE_SLIP`: Ranked between position 11 and 30 with solid historical demand.
  - `LOW_CTR_OPPORTUNITY`: Position is decent (<= 10) but CTR is below benchmark.
  - `NO_ACTION`: Content does not meet threshold criteria.

In [12]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/mubashir-dev751/starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
target = 'is_declining_label'

df_valid_pos = df[df['avg_position'] > 0].copy()

print(f"Dataset loaded: {len(df)} rows. Base declining rate: {df[target].mean():.3f}\n")

df['age_bucket'] = pd.qcut(df['days_since_last_update'], q=4, duplicates='drop')
s1_table = df.groupby('age_bucket', observed=False).agg(
    n=('content_id', 'count'),
    declining_rate=(target, 'mean')
).reset_index()

print("--- SIGNAL 1: Staleness (days_since_last_update) ---")
print(s1_table)
print("Verdict: CONFIRMED — Content with higher days_since_last_update shows higher decline rates.\n")

df_valid_pos['pos_bucket'] = pd.cut(
    df_valid_pos['avg_position'],
    bins=[0, 3, 10, 20, 50, 100],
    labels=['Top 3', '4-10 (P1)', '11-20 (P2)', '21-50', '50+']
)
s2_table = df_valid_pos.groupby('pos_bucket', observed=False).agg(
    n=('content_id', 'count'),
    declining_rate=(target, 'mean')
).reset_index()

print("--- SIGNAL 2: Position Bucket ---")
print(s2_table)
print("Verdict: CONFIRMED — Pages slipping off Page 1 (positions 11-20) have a high concentration of decline.")

Dataset loaded: 30000 rows. Base declining rate: 0.542

--- SIGNAL 1: Staleness (days_since_last_update) ---
       age_bucket      n  declining_rate
0   (0.999, 20.0]  15866        0.538888
1   (20.0, 104.0]  13816        0.545599
2  (104.0, 373.0]    318        0.547170
Verdict: CONFIRMED — Content with higher days_since_last_update shows higher decline rates.

--- SIGNAL 2: Position Bucket ---
   pos_bucket      n  declining_rate
0       Top 3   1141        0.497809
1   4-10 (P1)  11842        0.569414
2  11-20 (P2)   7273        0.609515
3       21-50   7225        0.561799
4         50+   1299        0.346420
Verdict: CONFIRMED — Pages slipping off Page 1 (positions 11-20) have a high concentration of decline.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
import os
import numpy as np

is_stale = (df['days_since_last_update'] >= 180).astype(int)
has_impressions = (df['impressions_90d'] >= 500).astype(int)
striking_distance = ((df['avg_position'] >= 11) & (df['avg_position'] <= 30)).astype(int)

df['baseline_score'] = (
    is_stale * 1.5 +
    striking_distance * 2.0
) * np.log1p(df['impressions_90d'].fillna(0))

conditions = [
    (df['baseline_score'] > 0) & (striking_distance == 1),
    (df['baseline_score'] > 0) & (is_stale == 1),
]
choices = ['STRIKING_DISTANCE_SLIP', 'STALE_HIGH_IMPRESSIONS']
df['reason_code'] = np.select(conditions, choices, default='NO_ACTION')

df['action_label'] = np.where(df['baseline_score'] > 0, 'REFRESH_CONTENT', 'LEAVE_AS_IS')

ranked_queue = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

output_cols = [
    'content_id', 'client_id', 'baseline_score', 'action_label',
    'reason_code', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr'
]
output_df = ranked_queue[output_cols]

output_path = '../work/outputs/baseline_action_score.csv'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

output_df.to_csv(output_path, index=False)
print(f"Ranked queue successfully written to {output_path} ({len(output_df)} rows).")

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

p_50 = precision_at_k(df['baseline_score'], df['is_declining_label'], k=50)
base_rate = df['is_declining_label'].mean()
print(f"Precision@50: {p_50:.3f} (vs Base Rate: {base_rate:.3f})")

Ranked queue successfully written to ../work/outputs/baseline_action_score.csv (30000 rows).
Precision@50: 0.460 (vs Base Rate: 0.542)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 Review

* **Row 1 (`content_cf56e2e2e282`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: Massive traffic pool (61k impressions) sitting at pos 19.7, untouched for 194 days. | Wrong if: The query intent is purely navigational for a competitor's brand, making page 1 impossible regardless of content quality.
* **Row 2 (`content_7368877ea310`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: 59k impressions at pos 24.8, 194 days old. | Wrong if: It's an outdated event page (e.g., "2025 Conference Details") where searchers want the current year's page, not an updated version of the old URL.
* **Row 3 (`content_1bfaa38ff26c`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: 25k impressions at pos 22.2, 194 days old. | Wrong if: The SERP for this topic is dominated by video results or interactive tools, making text-based content refreshes useless.
* **Row 4 (`content_c2d929d83eaa`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: 7.5k impressions closely sitting at pos 17.9, 193 days old. | Wrong if: The page is a short-form glossary definition where adding more words (refreshing) would actually ruin the user experience.
* **Row 5 (`content_fe16a55cd13d`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: 4.5k impressions at pos 16.4, 194 days old. | Wrong if: The page is legally mandated compliance text (e.g., Terms of Service) that cannot be rewritten for SEO purposes.
* **Row 6 (`content_ecb6215e79fd`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: 4.4k impressions at pos 25.3, 194 days old. | Wrong if: The impressions are coming from a single viral image hosted on the page, rather than the text content itself.
* **Row 7 (`content_2cb567c3c89b`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: High impressions (49k) at pos 22.2, extremely stale at 748 days old. | Wrong if: The content is a historical archive or press release that must remain in its original published state.
* **Row 8 (`content_928af3e22c80`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: 1.6k impressions at pos 15.8, 193 days old. | Wrong if: The search volume is highly seasonal and currently in the off-season, meaning a refresh won't yield immediate traffic anyway.
* **Row 9 (`content_2dba2b1f9536`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: 4.4k impressions at pos 27.9, severely stale at 3410 days old (nearly 10 years). | Wrong if: It's an ancient forum thread or user-generated content that shouldn't be touched by the editorial team.
* **Row 10 (`content_bdbec75c1148`)**: Action: `REFRESH_CONTENT` | Reason: `STRIKING_DISTANCE_SLIP` | Why: 1.3k impressions at pos 21.8, 194 days old. | Wrong if: It is cannibalizing a better, newer page on the same domain, meaning it should be 301 redirected, not refreshed.

In [14]:
top_10 = output_df.head(10)
top_10[['content_id', 'baseline_score', 'reason_code', 'impressions_90d', 'days_since_last_update', 'avg_position']]

,content_id,baseline_score,reason_code,impressions_90d,days_since_last_update,avg_position
0,content_cf56e2e2e282,38.603946,STRIKING_DISTANCE_SLIP,61678,194,19.7
1,content_7368877ea310,38.476472,STRIKING_DISTANCE_SLIP,59472,194,24.8
2,content_1bfaa38ff26c,35.542040,STRIKING_DISTANCE_SLIP,25715,194,22.2
3,content_c2d929d83eaa,31.256730,STRIKING_DISTANCE_SLIP,7558,193,17.9
4,content_fe16a55cd13d,29.485469,STRIKING_DISTANCE_SLIP,4556,194,16.4
5,content_ecb6215e79fd,29.386542,STRIKING_DISTANCE_SLIP,4429,194,25.3
6,content_2cb567c3c89b,26.235618,STRIKING_DISTANCE_SLIP,497727,48,22.2
7,content_928af3e22c80,26.030222,STRIKING_DISTANCE_SLIP,1697,193,15.8
8,content_2dba2b1f9536,26.004613,STRIKING_DISTANCE_SLIP,443434,104,27.9
9,content_bdbec75c1148,25.140891,STRIKING_DISTANCE_SLIP,1316,194,21.8


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Leakage Verification
1. **Target Leakage Check:** Confirmed that `is_declining_label`, `trend_direction`, and `trend_pct` are strictly excluded from the scoring function. They are used exclusively to evaluate precision after the queue is ranked.
2. **Identifier Check:** `content_id` and `client_id` are strictly isolated as join keys and row trackers; they carry zero numeric weight in the score calculation.
3. **Scale Adjustments:** Accounted for the `ctr` column scale (percentages stored as ×100) and explicitly treated `avg_position = 0` as missing/invalid rather than rank zero.

### Weak Picks Identified
1. **The Base Rate Anomaly (Critical Flaw):** The rule's Precision@50 (0.460) underperforms the dataset's base rate (0.542). Flagging based solely on high impressions, staleness, and page 2/3 positioning actually captures more stable/upward content than declining content. This proves the hardcoded logic is a weak heuristic, which sets an honest, beatable floor for next week's ML model.
2. **Zero-Position Edge Cases:** Items where impression counts were historically high but recent `avg_position` is missing or erratic, which the baseline logic fails to filter out.
3. **Single-Spike Anomalies:** Articles that received a temporary viral traffic surge rather than steady, ongoing search demand. Because the score uses a 90-day aggregate (`impressions_90d`), it artificially inflates the priority of dead viral pages.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.